# 06 — Domain Feature Engineering & Leakage Firewall

## Loan Default Risk Classification


## 1. Imports and Configuration


In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

# Paths
DATA_CLEAN_PATH = os.path.join('..', '..', 'data', 'processed', 'loan_cleaned.xlsx')
if not os.path.exists(DATA_CLEAN_PATH):
    DATA_CLEAN_PATH = os.path.join('data', 'processed', 'loan_cleaned.xlsx')

DATA_FEATURED_PATH = os.path.join('..', '..', 'data', 'processed', 'loan_featured.xlsx')
if not os.path.exists(os.path.dirname(DATA_FEATURED_PATH)):
    DATA_FEATURED_PATH = os.path.join('data', 'processed', 'loan_featured.xlsx')

print(f'Cleaned data source: {DATA_CLEAN_PATH}')
print(f'Featured data target: {DATA_FEATURED_PATH}')

Cleaned data source: ..\..\data\processed\loan_cleaned.xlsx
Featured data target: ..\..\data\processed\loan_featured.xlsx


## 2. Ingesting Cleaned Dataset


In [2]:
df = pd.read_excel(DATA_CLEAN_PATH, sheet_name='Loan_Cleaned')
print(f'Cleaned data loaded: {df.shape[0]:,} rows x {df.shape[1]} columns.')

Cleaned data loaded: 10,000 rows x 43 columns.


## 3. Financial Ratio Engineering: Debt-to-Income (DTI)


In [3]:
# 1. Monthly interest rate
monthly_rate = (df['Interest_Rate_Pct'] / 100.0) / 12.0
n_months = df['Loan_Term_Months']
principal = df['Loan_Amount']

# 2. Standard annuity PMT instalment formula
instalment = principal * (monthly_rate * (1.0 + monthly_rate)**n_months) / ((1.0 + monthly_rate)**n_months - 1.0)
df['New_Loan_Instalment'] = instalment

# 3. Monthly income (safe division)
monthly_income = df['Annual_Income'] / 12.0

# 4. Total Debt-to-Income ratio
df['Debt_to_Income'] = (df['Monthly_Debt_Payment'] + df['New_Loan_Instalment']) / monthly_income

# Display the first 5 rows
pmt_cols = ['Applicant_ID', 'Loan_Amount', 'Interest_Rate_Pct', 'Loan_Term_Months', 'New_Loan_Instalment', 'Monthly_Debt_Payment', 'Annual_Income', 'Debt_to_Income']
print('=== First 5 Applicants with Calculated PMT and Debt-to-Income ===')
display(df[pmt_cols].head(5))


=== First 5 Applicants with Calculated PMT and Debt-to-Income ===


,Applicant_ID,Loan_Amount,Interest_Rate_Pct,Loan_Term_Months,New_Loan_Instalment,Monthly_Debt_Payment,Annual_Income,Debt_to_Income
0,LN-114369,11500,10.29,12,1012.584440,320,51800.0,0.308707
1,LN-136223,11500,6.89,12,994.474510,190,NaN,NaN
2,LN-103022,10000,6.52,36,306.581069,215,18400.0,0.340162
3,LN-107320,12000,9.96,36,386.980928,35,20400.0,0.248224
4,LN-129811,22000,4.05,60,405.660077,460,45300.0,0.229314


## 4. Additional Lending Features & Validation


In [4]:
# 1. Loan_to_Income (LTI)
df['Loan_to_Income'] = df['Loan_Amount'] / df['Annual_Income']

# 2. Loan_to_Value (LTV) for Home loans (0 for non-home loans)
df['Loan_to_Value'] = np.where(df['Property_Value'] > 0, df['Loan_Amount'] / df['Property_Value'], 0.0)

# 3. Savings_to_Income (Financial cushion ratio)
df['Savings_to_Income'] = df['Savings_Balance'] / df['Annual_Income']

# 4. Log Transformations for skewed monetary features
df['Log_Annual_Income'] = np.log1p(df['Annual_Income'].clip(lower=0))
df['Log_Loan_Amount'] = np.log1p(df['Loan_Amount'].clip(lower=0))
df['Log_Savings_Balance'] = np.log1p(df['Savings_Balance'].clip(lower=0))

# 5. Temporal features from Application_Date
app_dates = pd.to_datetime(df['Application_Date'])
df['Application_Year'] = app_dates.dt.year
df['Application_Month'] = app_dates.dt.month
df['Application_Quarter'] = app_dates.dt.quarter

# 6. Categorical Bins
df['Age_Band'] = pd.cut(df['Age'], bins=[0, 30, 45, 60, 100], labels=['<30', '30-45', '45-60', '60+'])
df['Credit_Util_Band'] = pd.cut(df['Credit_Utilization_Pct'], bins=[-1, 30, 60, 80, 100], labels=['Low', 'Moderate', 'High', 'Severe'])

print('Engineered features summary:')
engineered_cols = ['Debt_to_Income', 'Loan_to_Income', 'Loan_to_Value', 'Savings_to_Income', 'Log_Annual_Income', 'Log_Loan_Amount']
display(df[engineered_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])


Engineered features summary:


,mean,std,min,50%,max
Debt_to_Income,0.342612,0.218261,0.011086,0.295726,4.144289
Loan_to_Income,0.689571,0.730090,0.019355,0.438596,16.428571
Loan_to_Value,0.136360,0.298321,0.000000,0.000000,1.000000
Savings_to_Income,0.565298,0.845557,0.000000,0.334504,29.329617
Log_Annual_Income,10.500955,0.595397,7.741099,10.529453,13.610945
Log_Loan_Amount,9.725866,0.917110,6.908755,9.680406,12.429220


### Empirical Validation: Evaluating AUC Improvement


In [5]:
# Benchmark Feature Sets with Stratified 5-Fold CV
y = (df['Loan_Default'] == 'Yes').astype(int)

# Set 1: Baseline raw numeric features
raw_features = ['Age', 'Annual_Income', 'Loan_Amount', 'Interest_Rate_Pct', 'Credit_Score', 'Credit_Utilization_Pct', 'Late_Payments_2Y', 'Monthly_Debt_Payment']
X_raw = df[raw_features].fillna(df[raw_features].median())

# Set 2: Enhanced with engineered ratios
enhanced_features = raw_features + ['Debt_to_Income', 'Loan_to_Income', 'Savings_to_Income', 'Log_Annual_Income', 'Log_Loan_Amount']
X_enhanced = df[enhanced_features].fillna(df[enhanced_features].median())

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler = StandardScaler()

# Evaluate Raw
scores_raw = cross_val_score(LogisticRegression(max_iter=1000), scaler.fit_transform(X_raw), y, cv=cv, scoring='roc_auc')

# Evaluate Enhanced
scores_enhanced = cross_val_score(LogisticRegression(max_iter=1000), scaler.fit_transform(X_enhanced), y, cv=cv, scoring='roc_auc')

print('=== Cross-Validated AUC Improvement Comparison ===')
print(f'Raw Features Baseline CV AUC:    {scores_raw.mean():.4f} +/- {scores_raw.std():.4f}')
print(f'Enhanced Features CV AUC:        {scores_enhanced.mean():.4f} +/- {scores_enhanced.std():.4f}')
print(f'Net AUC Improvement:             +{(scores_enhanced.mean() - scores_raw.mean()):.4f}')

# Save validation comparison table
df_auc_comp = pd.DataFrame({
    'Feature_Set': ['Raw Baseline Features', 'Enhanced with Engineered Ratios (DTI, LTI, etc.)'],
    'Mean_CV_AUC': [scores_raw.mean(), scores_enhanced.mean()],
    'Std_CV_AUC': [scores_raw.std(), scores_enhanced.std()],
    'Fold_Scores': [str(np.round(scores_raw, 4)), str(np.round(scores_enhanced, 4))]
})
df_auc_comp.to_csv('../../outputs/tables/feature_engineering_auc_comparison.csv', index=False)
display(df_auc_comp)

=== Cross-Validated AUC Improvement Comparison ===
Raw Features Baseline CV AUC:    0.8528 +/- 0.0101
Enhanced Features CV AUC:        0.8696 +/- 0.0103
Net AUC Improvement:             +0.0168


,Feature_Set,Mean_CV_AUC,Std_CV_AUC,Fold_Scores
0,Raw Baseline Features,0.852802,0.010128,[0.8398 0.8652 0.8576 0.8419 0.8596]
1,"Enhanced with Engineered Ratios (DTI, LTI, etc.)",0.869630,0.010332,[0.8555 0.8842 0.8714 0.8609 0.8761]


## 5. Categorical Variable Classification & Dummy Variable Trap


In [6]:
# Categorical classification inventory
cat_inventory = [
    {'Variable': 'Education_Level', 'Type': 'Ordinal', 'Levels': 5, 'Encoding_Strategy': 'OrdinalEncoder (0 to 4)', 'Reference_Dropped': 'N/A', 'Model_Columns': 1},
    {'Variable': 'Gender', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Mapping (0/1)', 'Reference_Dropped': 'Female', 'Model_Columns': 1},
    {'Variable': 'Salary_Account_With_Bank', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Mapping (0/1)', 'Reference_Dropped': 'No', 'Model_Columns': 1},
    {'Variable': 'Vehicle_Owned', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Mapping (0/1)', 'Reference_Dropped': 'No', 'Model_Columns': 1},
    {'Variable': 'Has_Co_Applicant', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Mapping (0/1)', 'Reference_Dropped': 'No', 'Model_Columns': 1},
    {'Variable': 'Previous_Default', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Mapping (0/1)', 'Reference_Dropped': 'No', 'Model_Columns': 1},
    {'Variable': 'Mobile_Banking_User', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Mapping (0/1)', 'Reference_Dropped': 'No', 'Model_Columns': 1},
    {'Variable': 'Ever_Delinquent', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Indicator (0/1)', 'Reference_Dropped': '0', 'Model_Columns': 1},
    {'Variable': 'Is_Home_Loan', 'Type': 'Binary', 'Levels': 2, 'Encoding_Strategy': 'Binary Indicator (0/1)', 'Reference_Dropped': '0', 'Model_Columns': 1},
    {'Variable': 'Marital_Status', 'Type': 'Nominal', 'Levels': 4, 'Encoding_Strategy': 'OneHotEncoder(drop=first)', 'Reference_Dropped': 'Divorced', 'Model_Columns': 3},
    {'Variable': 'Home_Ownership', 'Type': 'Nominal', 'Levels': 3, 'Encoding_Strategy': 'OneHotEncoder(drop=first)', 'Reference_Dropped': 'Mortgage', 'Model_Columns': 2},
    {'Variable': 'Employment_Type', 'Type': 'Nominal', 'Levels': 5, 'Encoding_Strategy': 'OneHotEncoder(drop=first)', 'Reference_Dropped': 'Contract', 'Model_Columns': 4},
    {'Variable': 'Employer_Sector', 'Type': 'Nominal', 'Levels': 9, 'Encoding_Strategy': 'OneHotEncoder(drop=first)', 'Reference_Dropped': 'Agriculture', 'Model_Columns': 8},
    {'Variable': 'Loan_Purpose', 'Type': 'Nominal', 'Levels': 6, 'Encoding_Strategy': 'OneHotEncoder(drop=first)', 'Reference_Dropped': 'Auto', 'Model_Columns': 5},
    {'Variable': 'Region', 'Type': 'Nominal', 'Levels': 3, 'Encoding_Strategy': 'OneHotEncoder(drop=first)', 'Reference_Dropped': 'Rural', 'Model_Columns': 2},
    {'Variable': 'Application_Channel', 'Type': 'Nominal', 'Levels': 3, 'Encoding_Strategy': 'OneHotEncoder(drop=first)', 'Reference_Dropped': 'Agent', 'Model_Columns': 2}
]

df_cat_inv = pd.DataFrame(cat_inventory)
df_cat_inv.to_csv('../../outputs/tables/categorical_encoding_inventory.csv', index=False)

print('=== Categorical Variable Encoding Inventory ===')
display(df_cat_inv)
print(f'Total resulting dummy/encoded feature columns: {df_cat_inv["Model_Columns"].sum()}')

=== Categorical Variable Encoding Inventory ===


,Variable,Type,Levels,Encoding_Strategy,Reference_Dropped,Model_Columns
0,Education_Level,Ordinal,5,OrdinalEncoder (0 to 4),N/A,1
1,Gender,Binary,2,Binary Mapping (0/1),Female,1
2,Salary_Account_With_Bank,Binary,2,Binary Mapping (0/1),No,1
3,Vehicle_Owned,Binary,2,Binary Mapping (0/1),No,1
4,Has_Co_Applicant,Binary,2,Binary Mapping (0/1),No,1
5,Previous_Default,Binary,2,Binary Mapping (0/1),No,1
6,Mobile_Banking_User,Binary,2,Binary Mapping (0/1),No,1
7,Ever_Delinquent,Binary,2,Binary Indicator (0/1),0,1
8,Is_Home_Loan,Binary,2,Binary Indicator (0/1),0,1
9,Marital_Status,Nominal,4,OneHotEncoder(drop=first),Divorced,3


Total resulting dummy/encoded feature columns: 35


## 6. Strict Leakage Firewall & Featured Dataset Export


In [7]:
# Save featured dataset
with pd.ExcelWriter(DATA_FEATURED_PATH, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Loan_Featured', index=False)

print(f'Successfully exported featured dataset to: {DATA_FEATURED_PATH}')
print(f'Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns.')

Successfully exported featured dataset to: ..\..\data\processed\loan_featured.xlsx
Dimensions: 10,000 rows x 56 columns.
